# Entrega 3 · Machine Learning: estimar la proteína de los productos sin datos

En la entrega 2 preparé los datos de Mercadona y de Open Food Facts hasta tener un Gold con los nutrientes de cada producto. El problema es que algunos productos no aparecen en Open Food Facts o tienen los datos incompletos, y sin nutrientes mi producto no puede recomendarlos.

En este notebook pruebo si un modelo de Machine Learning puede **estimar los gramos de proteína por 100 g** de esos productos usando solo lo que Mercadona sí publica (nombre, categoría, precio y formato). Es un problema de regresión.

**Qué hago, en orden:**
1. Cargo el Gold ampliado y reviso los datos.
2. Preparo las variables, sin fugas de información, y separo entrenamiento y test.
3. **Primera ronda:** comparo un baseline sencillo con dos modelos.
4. **Segunda ronda:** pruebo una variante que mejora la idea, eligiendo solo con validación cruzada.
5. Compruebo si la mejora es real, qué pasaría sin el control de fugas y qué rango de error tiene la estimación.
6. Fijo la regla y la fiabilidad por categoría.
7. Confirmo la comparación con un **test nuevo**, de productos que no había visto.
8. Compruebo la estimación con las **etiquetas reales** de productos sin datos.
9. Añado el **estado** (seco, cocinado…) a la regla y genero la salida para el producto.
10. Conclusión.

**Para reproducirlo:** primero hay que ejecutar `pipeline_ampliado.ipynb`, que genera `data/gold/productos_gold_ampliado.csv`, `data/silver/productos_silver_ampliado.csv` y `data/gold/test_nuevo.csv`. Los datos no se publican en el repositorio; las métricas sí están en `resultados/`.

## 1. Cargar el Gold ampliado y mirar los datos

Cargo el Gold ampliado, que es el de la entrega 2 llevado de 3 a 43 subcategorías de alimentos envasados (1037 productos en lugar de 70). Con 70 productos un modelo no se puede validar de forma seria.

Antes de entrenar nada compruebo tres cosas:
- **Qué columnas tienen huecos** entre las que usaría como entrada.
- **Qué productos tienen la proteína más alta**, por si hay errores de origen.
- **Si hay productos con proteína 0** en el grupo base, que es raro en alimentos como queso o embutido.

In [1]:
import pandas as pd

gold = pd.read_csv("data/gold/productos_gold_ampliado.csv", dtype={"ean": str})
print(gold.shape)

print("\nNulos en las columnas que usaría el modelo como entrada:")
entradas = ["nombre", "categoria_id", "grupo", "formato", "estado",
            "precio_unidad", "precio_kg", "tamano_envase"]
print(gold[entradas].isna().sum())

print("\nProteína más alta (posibles errores o productos especiales):")
cols = ["nombre", "grupo", "proteina_100g", "kcal_100g", "estado_calidad"]
print(gold.sort_values("proteina_100g", ascending=False)[cols].head(15).to_string())

print("\nProteína 0 en el grupo base:")
print(gold[(gold["grupo"] == "base") & (gold["proteina_100g"] == 0)][cols].head(15).to_string())

(1037, 22)

Nulos en las columnas que usaría el modelo como entrada:
nombre             0
categoria_id       0
grupo              0
formato          114
estado             0
precio_unidad      0
precio_kg          7
tamano_envase      0
dtype: int64

Proteína más alta (posibles errores o productos especiales):
                                                                     nombre        grupo  proteina_100g   kcal_100g  estado_calidad
296                Torreznillos Hacendado cortezas de cerdo fritas ahumadas  snack_dulce           59.0  570.000000       coherente
283                                                     Torreznos Hacendado  snack_dulce           52.0  600.000000       coherente
23                                                  Soja texturizada Luengo         base           50.0  361.000000       coherente
237     Cacahuete en polvo desgrasado +Proteínas 14 g 70% reducido en grasa         base           46.7  416.666667       coherente
281                       Co

## 2. Preparar los datos

**Qué vi en el paso anterior:**
- `formato` tiene 114 huecos. Los relleno con "desconocido" en vez de descartar esos productos.
- Las proteínas más altas (torreznos, cortezas, soja texturizada, jamón) son valores reales y los dejo.
- Un producto sí es un error: **"Queso curado mezcla con trufa Hacendado"** aparece con 0 g de proteína y 900 kcal, es decir, grasa pura. Un queso no es así. Es un error de Open Food Facts que el control de calorías de la entrega 2 no detecta, porque la comprobación 4/4/9 cuadra con esos números. Lo excluyo del entrenamiento y lo dejo aquí escrito.

**Qué variables uso:** solo lo que Mercadona publica antes de conocer los nutrientes: categoría, nombre, precios, tamaño, formato, estado y si es Hacendado. Para el precio por kilo uso `precio_kg`. Los pocos productos cuyo precio es por unidad o docena no tienen precio por kilo, y ese hueco se rellena con la mediana **dentro del pipeline, calculada solo con los datos de entrenamiento de cada fold**. Antes la calculaba con todo el Gold antes de separar los datos, lo que era una fuga pequeña. Ya está corregida y los resultados no cambiaron. **No uso calorías, grasa ni hidratos como entrada**, porque un producto sin datos en Open Food Facts tampoco los tiene, y además estarían muy ligados a la proteína (sería una fuga de información).

**Variante del producto.** Creo una columna con las dos primeras palabras del nombre sin la marca (por ejemplo, "pipas girasol"). La uso más adelante para que productos casi iguales no queden uno en entrenamiento y otro en test.

In [2]:
import numpy as np

# 1) Exclusión documentada: el queso con trufa tiene 0 g de proteína y 900 kcal,
#    que es un error de Open Food Facts (un queso no es grasa pura)
mask_error = (gold["nombre"] == "Queso curado mezcla con trufa Hacendado") & (gold["proteina_100g"] == 0)
print("Productos excluidos del entrenamiento por error de origen:", mask_error.sum())
print(gold.loc[mask_error, ["ean", "nombre", "kcal_100g", "proteina_100g", "grasa_100g", "hidratos_100g"]].to_string())
datos = gold[~mask_error].copy()

# 2) Variables de entrada: solo lo que Mercadona publica ANTES de conocer los nutrientes
datos["categoria_id"] = datos["categoria_id"].astype(str)
datos["formato"] = datos["formato"].fillna("desconocido")
datos["es_hacendado"] = datos["nombre"].str.contains("hacendado", case=False).astype(int)
datos["log_precio_ref"] = np.log1p(datos["precio_kg"])
datos["log_precio_ud"] = np.log1p(datos["precio_unidad"])
datos["log_tamano"] = np.log1p(datos["tamano_envase"])

# 3) Grupo de variante: las dos primeras palabras del nombre sin la marca.
#    Sirve para que variantes casi iguales no queden una en train y otra en test (fuga de información)
datos["variante"] = (datos["nombre"].str.lower().str.replace("hacendado", "", regex=False)
                     .str.split().str[:2].str.join(" "))
print("\nProductos:", len(datos), "| variantes distintas:", datos["variante"].nunique())
print(datos["variante"].value_counts().head(8))

Productos excluidos del entrenamiento por error de origen: 1
               ean                                   nombre  kcal_100g  proteina_100g  grasa_100g  hidratos_100g
501  8480000048837  Queso curado mezcla con trufa Hacendado      900.0            0.0       100.0            0.0

Productos: 1036 | variantes distintas: 580
variante
pan de                21
barritas de           18
salchichas cocidas    14
queso fresco          13
patatas fritas        11
chocolate con         10
postre lácteo         10
chocolate negro       10
Name: count, dtype: int64


## 3. Primera ronda: baseline y dos modelos

**Separación de datos.** 80 % para entrenar y 20 % para test, agrupando por variante. Así el modelo no ve en test una variante casi idéntica a la que vio entrenando.

**Baseline.** Predigo, para cada producto, la proteína media de su categoría (calculada solo con el entrenamiento). Es lo que haría una persona sin ML: un yogur no tiene la proteína de un jamón. Cualquier modelo tiene que ganarle para justificar su complejidad.

**Modelos de ML:**
- **Ridge:** regresión lineal con penalización, buena con pocos datos y muchas columnas de texto.
- **Random Forest:** un modelo no lineal, por si hay relaciones que Ridge no capta.

**Cómo comparo.** Con validación cruzada de 5 pliegues, también agrupada por variante, **solo sobre el entrenamiento**. El test queda para el final.

**Métricas.** El **MAE** es el error medio en gramos por 100 g (la más fácil de entender). El RMSE penaliza más los errores grandes. El R² dice qué parte de la variación explica el modelo. `pct_error_<2g` es el porcentaje de productos con error menor de 2 g.

**Qué mirar en la salida:** si algún modelo baja el MAE del baseline y si la validación cruzada y el test cuentan la misma historia.

In [3]:
import os
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

OBJETIVO = "proteina_100g"

# --- Separación: 80 % entrenamiento / 20 % test, sin mezclar variantes entre ambos ---
idx_tr, idx_te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                      .split(datos, groups=datos["variante"]))
train, test = datos.iloc[idx_tr], datos.iloc[idx_te]
print("Entrenamiento:", len(train), "| Test:", len(test))

def metricas(real, pred):
    err = pred - real
    return {"MAE": np.abs(err).mean(),
            "RMSE": np.sqrt((err ** 2).mean()),
            "R2": 1 - (err ** 2).sum() / ((real - real.mean()) ** 2).sum(),
            "pct_error_<2g": (np.abs(err) < 2).mean() * 100}

# --- Baseline: proteína media de la categoría (con la media global si la categoría no existe) ---
def predecir_baseline(entrenamiento, df):
    media_cat = entrenamiento.groupby("categoria_id")[OBJETIVO].mean()
    return df["categoria_id"].map(media_cat).fillna(entrenamiento[OBJETIVO].mean()).values

# --- Modelos de ML ---
def preprocesado():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["categoria_id", "estado", "formato"]),
        ("txt", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "nombre"),
        # La mediana para los huecos se calcula solo con los datos de entrenamiento de cada fold
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("esc", StandardScaler())]),
         ["es_hacendado", "log_precio_ref", "log_precio_ud", "log_tamano"]),
    ])

modelos = {
    "Ridge": lambda: Pipeline([("pre", preprocesado()), ("m", Ridge(alpha=1.0))]),
    "Random Forest": lambda: Pipeline([("pre", preprocesado()),
                                       ("m", RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                                                   n_jobs=-1, random_state=42))]),
}

# --- Validación cruzada agrupada SOLO sobre entrenamiento (para comparar modelos) ---
cv = GroupKFold(n_splits=5)
cv_mae = {"Baseline (media por categoría)": []}
cv_mae.update({n: [] for n in modelos})
for a, b in cv.split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a], train.iloc[b]
    cv_mae["Baseline (media por categoría)"].append(
        metricas(va_f[OBJETIVO].values, predecir_baseline(tr_f, va_f))["MAE"])
    for n, f in modelos.items():
        m = f().fit(tr_f, tr_f[OBJETIVO])
        cv_mae[n].append(metricas(va_f[OBJETIVO].values, m.predict(va_f))["MAE"])

# --- Evaluación final en test (datos que el modelo no ha visto) ---
pred_test = {"Baseline (media por categoría)": predecir_baseline(train, test)}
for n, f in modelos.items():
    pred_test[n] = f().fit(train, train[OBJETIVO]).predict(test)

filas = []
for n, p in pred_test.items():
    r = metricas(test[OBJETIVO].values, p)
    r.update({"modelo": n, "MAE_cv_media": np.mean(cv_mae[n]), "MAE_cv_desv": np.std(cv_mae[n])})
    filas.append(r)
resultados = pd.DataFrame(filas).set_index("modelo").round(2)
print(resultados.to_string())

os.makedirs("resultados", exist_ok=True)
resultados.to_csv("resultados/metricas_ml.csv")

# --- Análisis de errores del mejor modelo según validación cruzada ---
mejor = min(modelos, key=lambda n: np.mean(cv_mae[n]))
print("\nMejor modelo por validación cruzada:", mejor)
t = test.copy()
t["pred"] = pred_test[mejor]
t["pred_baseline"] = pred_test["Baseline (media por categoría)"]
t["error_abs"] = (t["pred"] - t[OBJETIVO]).abs()
t["error_abs_baseline"] = (t["pred_baseline"] - t[OBJETIVO]).abs()

print("\nError medio (g) por grupo, modelo vs baseline:")
print(t.groupby("grupo")[["error_abs", "error_abs_baseline"]].mean().round(2))
print("\nError medio (g) Hacendado (1) vs otras marcas (0):")
print(t.groupby("es_hacendado")[["error_abs", "error_abs_baseline"]].agg(["mean", "count"]).round(2))
print("\n10 mayores errores:")
print(t.sort_values("error_abs", ascending=False)[["nombre", "grupo", OBJETIVO, "pred"]].head(10).round(1).to_string())

Entrenamiento: 849 | Test: 187
                                 MAE  RMSE    R2  pct_error_<2g  MAE_cv_media  MAE_cv_desv
modelo                                                                                    
Baseline (media por categoría)  3.39  5.14  0.60          48.13          3.20         0.21
Ridge                           3.41  4.91  0.64          42.25          3.24         0.22
Random Forest                   3.67  5.76  0.50          48.13          3.64         0.37

Mejor modelo por validación cruzada: Ridge

Error medio (g) por grupo, modelo vs baseline:
                 error_abs  error_abs_baseline
grupo                                         
base                  4.06                4.01
plato_preparado       2.19                2.80
salsa                 2.20                1.34
snack_dulce           2.61                2.58

Error medio (g) Hacendado (1) vs otras marcas (0):
             error_abs       error_abs_baseline      
                  mean count      

## 4. Segunda ronda: la categoría hace el trabajo grueso, el modelo afina

**Lo que vi en la primera ronda.** Ridge y Random Forest no mejoraban al baseline. Ridge intenta aprender la proteína desde cero y no aprovecha bien que la categoría ya explica gran parte.

**La idea nueva.** Un modelo "baseline + corrección": parte de la media de la categoría y Ridge solo aprende cuánto se desvía un producto de esa media según su nombre y su precio. Pruebo 5 valores de la penalización (alpha) para esta variante y para Ridge solo.

**Cómo evito engañarme.** Como ya vi el test en la primera ronda, esta vez **elijo el modelo solo con validación cruzada** sobre el entrenamiento. El test se mira una única vez, para el modelo elegido y el baseline.

**Qué mirar en la salida:** la tabla de validación cruzada (menor MAE = mejor), la comparación final en test, el porcentaje de productos donde el modelo acierta más que el baseline y dónde están los errores grandes.

## 5. ¿Es real la mejora? Fugas y rango de error

La mejora de la segunda ronda es pequeña. Antes de darla por buena compruebo cuatro cosas:

1. **¿La mejora se distingue del azar?** Repito miles de veces el test remuestreando los productos (bootstrap) y miro el intervalo de la mejora media. Si incluye el 0, no puedo asegurar que el modelo sea mejor.
2. **¿Cuánto cambiaría el resultado sin el control de fugas?** Comparo la validación agrupada por variante con una mezclada. Si la mezclada sale mucho mejor, queda demostrado que sin el control los resultados serían falsamente buenos.
3. **¿Qué rango de error acompañará a cada estimación?** Con las predicciones "fuera de pliegue" del entrenamiento calculo el rango en el que cae el 80 % de los errores.
4. **¿En qué categorías es poco fiable?** El error medio por categoría me dirá dónde no conviene estimar.

In [4]:
from sklearn.base import BaseEstimator, RegressorMixin

class ResidualSobreBaseline(BaseEstimator, RegressorMixin):
    """Predice la media de la categoría + una corrección aprendida (Ridge) a partir del nombre y el precio."""
    def __init__(self, modelo=None):
        self.modelo = modelo
    def _base(self, X):
        return X["categoria_id"].map(self.media_cat_).fillna(self.media_global_).values
    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.media_cat_ = y.groupby(X["categoria_id"]).mean()
        self.media_global_ = y.mean()
        self.modelo.fit(X, y.values - self._base(X))
        return self
    def predict(self, X):
        return np.clip(self._base(X) + self.modelo.predict(X), 0, 100)

def ridge(alpha):
    return Pipeline([("pre", preprocesado()), ("m", Ridge(alpha=alpha))])

candidatos = {}
for a in [0.3, 1, 3, 10, 30]:
    candidatos[f"Ridge (alpha={a})"] = (lambda a=a: ridge(a))
    candidatos[f"Baseline + corrección Ridge (alpha={a})"] = (lambda a=a: ResidualSobreBaseline(ridge(a)))

# --- Elección SOLO con validación cruzada sobre entrenamiento ---
cv2 = {"Baseline (media por categoría)": []}
cv2.update({n: [] for n in candidatos})
for a_idx, b_idx in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_idx], train.iloc[b_idx]
    cv2["Baseline (media por categoría)"].append(
        metricas(va_f[OBJETIVO].values, predecir_baseline(tr_f, va_f))["MAE"])
    for n, f in candidatos.items():
        m = f().fit(tr_f, tr_f[OBJETIVO])
        cv2[n].append(metricas(va_f[OBJETIVO].values, m.predict(va_f))["MAE"])

tabla_cv = pd.DataFrame({"MAE_cv_media": {n: np.mean(v) for n, v in cv2.items()},
                         "MAE_cv_desv": {n: np.std(v) for n, v in cv2.items()}}).round(2).sort_values("MAE_cv_media")
print("Validación cruzada (menor MAE = mejor):")
print(tabla_cv.to_string())

mejor_n = min(candidatos, key=lambda n: np.mean(cv2[n]))
print("\nModelo elegido por validación cruzada:", mejor_n)

# --- Evaluación final en test: una sola vez, solo el elegido y el baseline ---
modelo_final = candidatos[mejor_n]().fit(train, train[OBJETIVO])
t = test.copy()
t["pred"] = modelo_final.predict(test)
t["pred_baseline"] = predecir_baseline(train, test)
t["error_abs"] = (t["pred"] - t[OBJETIVO]).abs()
t["error_abs_baseline"] = (t["pred_baseline"] - t[OBJETIVO]).abs()

final = pd.DataFrame({
    "Baseline (media por categoría)": metricas(t[OBJETIVO].values, t["pred_baseline"].values),
    mejor_n: metricas(t[OBJETIVO].values, t["pred"].values)}).T.round(2)
print("\nTEST (datos no vistos):")
print(final.to_string())
print("\n% de productos donde el modelo se equivoca menos que el baseline:",
      round((t["error_abs"] < t["error_abs_baseline"]).mean() * 100, 1))

os.makedirs("resultados", exist_ok=True)
tabla_cv.to_csv("resultados/validacion_cruzada.csv")
final.to_csv("resultados/test_final.csv")

print("\nError medio (g) por grupo:")
print(t.groupby("grupo")[["error_abs", "error_abs_baseline"]].mean().round(2))
print("\nError medio (g) Hacendado (1) vs otras marcas (0):")
print(t.groupby("es_hacendado")[["error_abs", "error_abs_baseline"]].agg(["mean", "count"]).round(2))
print("\nCategorías con más error total en test:")
print(t.groupby("categoria_id")[["error_abs", "error_abs_baseline"]].agg(["mean", "count"]).round(2)
      .sort_values(("error_abs", "mean"), ascending=False).head(8))
print("\n10 mayores errores:")
print(t.sort_values("error_abs", ascending=False)[["nombre", "grupo", OBJETIVO, "pred"]].head(10).round(1).to_string())

Validación cruzada (menor MAE = mejor):
                                         MAE_cv_media  MAE_cv_desv
Baseline + corrección Ridge (alpha=3)            2.96         0.20
Baseline + corrección Ridge (alpha=10)           2.99         0.18
Baseline + corrección Ridge (alpha=1)            3.04         0.19
Baseline + corrección Ridge (alpha=30)           3.08         0.18
Baseline (media por categoría)                   3.20         0.21
Ridge (alpha=1)                                  3.24         0.22
Baseline + corrección Ridge (alpha=0.3)          3.25         0.12
Ridge (alpha=3)                                  3.29         0.26
Ridge (alpha=0.3)                                3.40         0.09
Ridge (alpha=10)                                 3.62         0.26
Ridge (alpha=30)                                 4.13         0.28

Modelo elegido por validación cruzada: Baseline + corrección Ridge (alpha=3)

TEST (datos no vistos):
                                        MAE  RMSE    

In [5]:
from sklearn.model_selection import KFold

nombres_cat = {"118": "Arroz", "121": "Legumbres", "120": "Pasta", "78": "Cereales", "80": "Galletas",
    "79": "Tortitas", "135": "Aceitunas y encurtidos", "133": "Frutos secos", "132": "Patatas fritas y snacks",
    "92": "Chocolate", "90": "Mermelada y miel", "48": "Aves y jamón cocido", "52": "Bacón y salchichas",
    "49": "Chopped y mortadela", "51": "Embutido curado", "50": "Jamón serrano", "58": "Paté y sobrasada",
    "54": "Queso curado", "56": "Queso lonchas y rallado", "53": "Queso untable y fresco", "43": "Embutido",
    "44": "Hamburguesas", "122": "Atún y conservas pescado", "123": "Berberechos y mejillones",
    "127": "Conservas verdura y fruta", "126": "Tomate", "103": "Yogures desnatados", "109": "Yogures griegos",
    "104": "Yogures naturales", "110": "Flan y natillas", "111": "Gelatina y postres", "77": "Huevos",
    "75": "Mantequilla y margarina", "60": "Pan de molde", "62": "Pan tostado y rallado",
    "64": "Picos y rosquilletas", "66": "Bollería envasada", "65": "Bollería de horno",
    "116": "Mayonesa, ketchup y mostaza", "117": "Otras salsas", "138": "Pizzas",
    "140": "Platos calientes", "142": "Platos fríos"}

# 1) ¿La mejora sobre el baseline es distinguible del azar? (bootstrap sobre el test)
rng = np.random.default_rng(42)
dif = (t["error_abs_baseline"] - t["error_abs"]).values     # > 0 = el modelo acierta más
boot = np.array([rng.choice(dif, size=len(dif), replace=True).mean() for _ in range(5000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Mejora media del modelo sobre el baseline: {dif.mean():.2f} g (intervalo 95 %: {lo:.2f} a {hi:.2f})")
print("  -> si el intervalo incluye el 0, la mejora no se puede distinguir del azar")

# 2) ¿Cuánto cambiaría el resultado SIN el control de fugas? (mezclando variantes entre train y test)
def mae_cv(splitter, **kw):
    r, b = [], []
    for a_i, b_i in splitter.split(train, **kw):
        tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
        m = candidatos[mejor_n]().fit(tr_f, tr_f[OBJETIVO])
        r.append(np.abs(m.predict(va_f) - va_f[OBJETIVO]).mean())
        b.append(np.abs(predecir_baseline(tr_f, va_f) - va_f[OBJETIVO]).mean())
    return np.mean(r), np.mean(b)
agr = mae_cv(GroupKFold(n_splits=5), groups=train["variante"])
mez = mae_cv(KFold(n_splits=5, shuffle=True, random_state=42))
print("\nMAE en validación cruzada (modelo / baseline):")
print(f"  Agrupada por variante (la que usamos): {agr[0]:.2f} / {agr[1]:.2f}")
print(f"  Mezclada (con fuga de variantes):      {mez[0]:.2f} / {mez[1]:.2f}")

# 3) Rango de error para el producto: residuos de predicciones "fuera de pliegue" en entrenamiento
oof = pd.Series(index=train.index, dtype=float)
for a_i, b_i in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
    oof.iloc[b_i] = candidatos[mejor_n]().fit(tr_f, tr_f[OBJETIVO]).predict(va_f)
residuo = train[OBJETIVO] - oof
q10, q90 = residuo.quantile([0.1, 0.9])
dentro = ((t[OBJETIVO] >= t["pred"] + q10) & (t[OBJETIVO] <= t["pred"] + q90)).mean() * 100
print(f"\nRango de error (80 %): estimación {q10:+.1f} g a {q90:+.1f} g")
print(f"Cobertura real en test (debería rondar el 80 %): {dentro:.1f} %")

# 4) ¿En qué categorías es poco fiable la estimación?
fiab = (train.assign(err=residuo.abs()).groupby("categoria_id")["err"]
        .agg(["mean", "count"]).query("count >= 8").sort_values("mean", ascending=False).round(1))
fiab.insert(0, "categoria", [nombres_cat.get(c, c) for c in fiab.index])
print("\nCategorías con peor error (validación cruzada, mínimo 8 productos):")
print(fiab.head(10).to_string())
print("\nCategorías con mejor error:")
print(fiab.tail(5).to_string())

Mejora media del modelo sobre el baseline: 0.03 g (intervalo 95 %: -0.18 a 0.24)
  -> si el intervalo incluye el 0, la mejora no se puede distinguir del azar

MAE en validación cruzada (modelo / baseline):
  Agrupada por variante (la que usamos): 2.96 / 3.20
  Mezclada (con fuga de variantes):      2.78 / 3.10

Rango de error (80 %): estimación -4.7 g a +3.6 g
Cobertura real en test (debería rondar el 80 %): 74.9 %

Categorías con peor error (validación cruzada, mínimo 8 productos):
                             categoria  mean  count
categoria_id                                       
132            Patatas fritas y snacks   9.1     29
133                       Frutos secos   6.5     35
121                          Legumbres   5.5     13
117                       Otras salsas   4.7     27
79                            Tortitas   4.4      8
56             Queso lonchas y rallado   4.2     18
122           Atún y conservas pescado   4.0     21
53              Queso untable y fresco   3.9

## 6. Regla elegida y fiabilidad por categoría

**Decisión.** Como la mejora del modelo con ML no se distingue del azar, me quedo con **el baseline: la media de proteína de la categoría**. Es más simple, se explica en una frase y no depende del nombre del producto, así que es menos probable que falle con productos nuevos.

**Cómo lo hago fiable:**
- Una categoría es **fiable** si tiene al menos 8 productos y su error en validación cruzada es de 4 g o menos. En el resto no se estima. Las categorías con menos de 8 productos en el entrenamiento tampoco se estiman.
- El **rango de error del 80 %** se calcula solo con las categorías fiables.
- Compruebo en test, que no se usó para decidir nada, que el rango cubre alrededor del 80 % de los productos.

**Aplicación.** Aplico la regla a los productos que no tienen datos nutricionales fiables y guardo la salida en `data/gold/estimaciones_proteina.csv`. Esta es la versión **solo por categoría**: la dejo como comparación, porque en la sección 9 añado el estado y esa es la regla final. Los resultados agregados (fiabilidad por categoría y cobertura de las estimaciones) se guardan en `resultados/`.

In [6]:
# 1) Errores fuera de pliegue del baseline en entrenamiento
oof_b = pd.Series(index=train.index, dtype=float)
for a_i, b_i in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
    oof_b.iloc[b_i] = predecir_baseline(tr_f, va_f)
res_b = train[OBJETIVO] - oof_b

# 2) Fiabilidad por categoría: al menos 8 productos y error medio <= 4 g
err_cat = res_b.abs().groupby(train["categoria_id"]).agg(["mean", "count"])
fiables = err_cat[(err_cat["count"] >= 8) & (err_cat["mean"] <= 4)].index
print("Categorías fiables:", len(fiables), "de", len(err_cat),
      "con productos en el entrenamiento (hay", len(nombres_cat), "en total)")
print("Categorías sin productos en el entrenamiento:",
      [nombres_cat[c] for c in nombres_cat if c not in err_cat.index])
print("Categorías NO fiables:", [nombres_cat.get(c, c) for c in err_cat.index.difference(fiables)])

# 3) Rango de error del 80 % usando solo categorías fiables
q10, q90 = res_b[train["categoria_id"].isin(fiables)].quantile([0.1, 0.9])
print(f"\nRango de error (80 %): {q10:+.1f} g a {q90:+.1f} g")

# 4) Comprobación en test (datos no vistos)
pred_b = predecir_baseline(train, test)
mask_te = test["categoria_id"].isin(fiables)
real = test[OBJETIVO]
dentro = ((real >= pred_b + q10) & (real <= pred_b + q90))[mask_te].mean() * 100
mae_fiables = np.abs(pred_b - real.values)[mask_te.values].mean()
mae_no_fiables = np.abs(pred_b - real.values)[~mask_te.values].mean()
print(f"Test, categorías fiables ({mask_te.sum()} productos): MAE {mae_fiables:.2f} g, "
      f"cobertura del rango {dentro:.1f} %")
print(f"Test, categorías no fiables ({(~mask_te).sum()} productos): MAE {mae_no_fiables:.2f} g")

# 5) Aplicación a los productos reales sin datos nutricionales fiables
silver_amp = pd.read_csv("data/silver/productos_silver_ampliado.csv", dtype={"ean": str})
sin = silver_amp[silver_amp["estado_calidad"] == "sin_datos"].copy()
sin["categoria_id"] = sin["categoria_id"].astype(str)
media_final = datos.groupby("categoria_id")[OBJETIVO].mean()

sin["proteina_estimada"] = sin["categoria_id"].map(media_final).round(1)
sin["fiable"] = sin["categoria_id"].isin(fiables)
sin["rango_bajo"] = (sin["proteina_estimada"] + q10).clip(lower=0).round(1)
sin["rango_alto"] = (sin["proteina_estimada"] + q90).round(1)
sin.loc[~sin["fiable"], ["proteina_estimada", "rango_bajo", "rango_alto"]] = np.nan

print("\nProductos sin datos nutricionales:", len(sin))
print("Con estimación fiable:", int(sin["fiable"].sum()), "| sin estimación:", int((~sin["fiable"]).sum()))
cobertura_est = sin.groupby("grupo")["fiable"].agg(con_estimacion="sum", total="count")
print(cobertura_est)

# Salida para el producto (no se publica: tiene nombres de productos)
salida = sin[["product_id", "ean", "nombre", "categoria_id", "grupo",
              "proteina_estimada", "rango_bajo", "rango_alto", "fiable"]]
salida.to_csv("data/gold/estimaciones_proteina.csv", index=False)

# Resultados agregados (sí se publican: solo cifras, sin datos de productos)
tabla_fiab = err_cat.round(2).rename(columns={"mean": "error_medio_g", "count": "productos"})
tabla_fiab["categoria"] = [nombres_cat.get(c, c) for c in tabla_fiab.index]
tabla_fiab["fiable"] = tabla_fiab.index.isin(fiables)
tabla_fiab.sort_values("error_medio_g").to_csv("resultados/fiabilidad_por_categoria.csv")
cobertura_est.to_csv("resultados/cobertura_estimaciones.csv")

print("\nEjemplos:")
print(salida[salida["fiable"]].head(8).to_string())

Categorías fiables: 26 de 42 con productos en el entrenamiento (hay 43 en total)
Categorías sin productos en el entrenamiento: ['Embutido']
Categorías NO fiables: ['Yogures naturales', 'Yogures griegos', 'Flan y natillas', 'Otras salsas', 'Legumbres', 'Patatas fritas y snacks', 'Frutos secos', 'Platos calientes', 'Platos fríos', 'Chopped y mortadela', 'Embutido curado', 'Queso curado', 'Queso lonchas y rallado', 'Bollería de horno', 'Huevos', 'Tortitas']

Rango de error (80 %): -3.4 g a +3.4 g
Test, categorías fiables (115 productos): MAE 2.08 g, cobertura del rango 82.6 %
Test, categorías no fiables (72 productos): MAE 5.48 g

Productos sin datos nutricionales: 269
Con estimación fiable: 120 | sin estimación: 149
                 con_estimacion  total
grupo                                 
base                         80    176
plato_preparado               2     18
salsa                         0      1
snack_dulce                  38     74

Ejemplos:
    product_id            ean  

## 7. Test nuevo: productos que no había visto

El test de la primera ronda ya lo había visto, así que no sirve para confirmar la comparación con honestidad. Descargué 10 categorías nuevas de Mercadona (platos listos, pizzas congeladas, rebozados, empanados, arroz y pasta congelados, salazones y ahumados, bífidus, postres de soja, sopas y turrones; la descarga está en `pipeline_ampliado.ipynb`). Quité los productos que ya tenía y los que tenían una variante ya presente en el entrenamiento, y quedaron **101 productos con proteína informada**. Entrené con todos mis datos anteriores y medí en estos, sin tocar nada del modelo.

Como estas categorías no existen en el entrenamiento, asigné cada una a una categoría parecida **antes de ver los resultados**. Es una limitación: algunas se parecen poco, y el test mide también si mi asignación fue buena.

| Modelo | MAE (g) | RMSE (g) | R² |
|---|---|---|---|
| Media por categoría | 4,26 | 6,00 | 0,01 |
| Ridge | 4,21 | 5,32 | 0,22 |
| Random Forest | 3,36 | 4,56 | 0,43 |

- **Ridge frente a la media por categoría:** son equivalentes (diferencia de -0,05 g, intervalo al 95 % de -0,56 a 0,47). Se confirma lo que veía en el test anterior.
- **Random Forest** mejora en 0,9 g (intervalo de -1,42 a -0,37), pero casi toda esa mejora viene de los rebozados congelados (24 de los 101 productos), donde mi asignación a «hamburguesas y picadas» es dudosa. Sin esa categoría, la diferencia baja a unos 0,2 g. En la comprobación con etiquetas reales (sección 8) Random Forest es el peor. Lo tomo como una pista a investigar, y no cambio de modelo después de ver este test.

**Conclusión:** mantengo la regla sencilla (la media por categoría) donde la categoría está bien representada, y uso el modelo con cautela donde no lo está.

In [7]:
# --- TEST NUEVO: productos que ni el modelo ni yo habíamos visto ---
test_n = pd.read_csv("data/gold/test_nuevo.csv", dtype={"ean": str})
test_n["categoria_id"] = test_n["categoria_id"].astype(str)
test_n["formato"] = test_n["formato"].fillna("desconocido")
test_n["es_hacendado"] = test_n["nombre"].str.contains("hacendado", case=False).astype(int)
test_n["log_precio_ref"] = np.log1p(test_n["precio_kg"])
test_n["log_precio_ud"] = np.log1p(test_n["precio_unidad"])
test_n["log_tamano"] = np.log1p(test_n["tamano_envase"])

y = test_n[OBJETIVO].values
print("Productos del test nuevo:", len(test_n))
print("Categorías del test que no están en el entrenamiento:",
      set(test_n["categoria_id"]) - set(datos["categoria_id"]))

# Se entrena con TODO lo que ya tenía (entrenamiento + test anterior) y se mide en lo nuevo
pred_n = {"Baseline (media por categoría)": predecir_baseline(datos, test_n)}
for nombre_m, f in modelos.items():
    pred_n[nombre_m] = f().fit(datos, datos[OBJETIVO]).predict(test_n)

filas = []
for nombre_m, p in pred_n.items():
    r = metricas(y, p)
    r["modelo"] = nombre_m
    filas.append(r)
res_nuevo = pd.DataFrame(filas).set_index("modelo").round(2)
print(res_nuevo.to_string())
res_nuevo.to_csv("resultados/metricas_test_nuevo.csv")

# ¿Es real la diferencia entre Ridge y la media por categoría? (remuestreo)
err_b = np.abs(pred_n["Baseline (media por categoría)"] - y)
err_r = np.abs(pred_n["Ridge"] - y)
dif = err_r - err_b
rng = np.random.default_rng(42)
boot = [rng.choice(dif, size=len(dif), replace=True).mean() for _ in range(5000)]
print("\nError de Ridge menos error de la media por categoría (g):", round(dif.mean(), 2))
print("Intervalo al 95 %:", np.percentile(boot, [2.5, 97.5]).round(2))

t2 = test_n.copy()
t2["err_baseline"] = err_b
t2["err_ridge"] = err_r
print("\nError medio (g) por grupo:")
print(t2.groupby("grupo")[["err_baseline", "err_ridge"]].agg(["mean", "count"]).round(2))


# Contraste de Random Forest con la media por categoría
err_rf = np.abs(pred_n["Random Forest"] - y)

dif_rf = err_rf - err_b
boot_rf = [rng.choice(dif_rf, size=len(dif_rf), replace=True).mean() for _ in range(5000)]
print("Error de Random Forest menos error de la media por categoría (g):", round(dif_rf.mean(), 2))
print("Intervalo al 95 %:", np.percentile(boot_rf, [2.5, 97.5]).round(2))

t2["err_rf"] = err_rf
print("\nError medio (g) por categoría de origen:")
print(t2.groupby("categoria_original")[["err_baseline", "err_ridge", "err_rf"]]
      .mean().round(2).assign(n=t2.groupby("categoria_original").size()))

Productos del test nuevo: 101
Categorías del test que no están en el entrenamiento: set()
                                 MAE  RMSE    R2  pct_error_<2g
modelo                                                         
Baseline (media por categoría)  4.26  6.00  0.01          44.55
Ridge                           4.21  5.32  0.22          33.66
Random Forest                   3.36  4.56  0.43          45.54

Error de Ridge menos error de la media por categoría (g): -0.05
Intervalo al 95 %: [-0.56  0.47]

Error medio (g) por grupo:
                err_baseline       err_ridge      
                        mean count      mean count
grupo                                             
base                    4.90    67      5.02    67
plato_preparado         3.12    14      3.34    14
snack_dulce             2.91    20      2.11    20
Error de Random Forest menos error de la media por categoría (g): -0.9
Intervalo al 95 %: [-1.42 -0.37]

Error medio (g) por categoría de origen:
            

## 8. Etiquetas reales de productos sin datos

Todo lo anterior mide el error en productos que sí tienen ficha nutricional. Los productos donde se usaría la estimación son justo los que no la tienen, y de esos nunca había visto el valor real. Para comprobarlo miré a mano la tabla nutricional (la fila de proteínas, por 100 g) en la web de Mercadona. Lo hice el 22 de noviembre de 2026 y en dos muestras:
- **Primera muestra:** 26 productos elegidos al azar entre los 269 sin datos.
- **Segunda muestra:** 27 productos más, elegidos al azar solo entre los que sí recibirían estimación (categorías fiables), porque con la primera quedaban pocos.

### 8.1 Primera muestra

Elegí al azar 26 de los 269 productos sin datos (semilla fija, repartidos según el tamaño de cada grupo). Uno (la empanadilla Cheese Burger) no tiene etiqueta y lo dejé fuera. Los fideos orientales los miro aparte, porque no sé si su etiqueta es del producto seco o del preparado (marca 2 g, así que parece el plato preparado). Quedan **24 productos** (15 del grupo base, 6 snacks y dulces, 2 platos preparados y 1 salsa). Entrené con todo el Gold y comparé la estimación con lo que pone la etiqueta:

| Estimación | Error medio (g) | Productos con error < 2 g |
|---|---|---|
| Media por categoría | 2,49 | 54 % |
| Ridge | 2,59 | 46 % |
| Random Forest | 4,58 | 38 % |

- La media por categoría y Ridge vuelven a ser equivalentes, y Random Forest es el peor: subestima mucho los quesos y los jamones. Esto contradice la ventaja que tuvo en el test nuevo, que dependía de una sola categoría, así que no la tomo como fiable.
- El mayor error de todos es el de los mini biscotes de trigo: la etiqueta marca 1,1 g (comprobado) y las estimaciones dan entre 8 y 10 g.

Solo 13 de estos 24 productos están en categorías fiables, que son los que recibirían una estimación: ahí el error medio es de 2,99 g y 9 de 13 (69 %) quedan dentro de ±3,4 g. Con 13 productos es una muestra demasiado pequeña para concluir nada, y por eso hice la segunda.

In [8]:
# Primera muestra: 26 productos sin datos, al azar y repartidos según el tamaño de cada grupo
silver_a = pd.read_csv("data/silver/productos_silver_ampliado.csv", dtype={"ean": str})
sin = silver_a[silver_a["estado_calidad"] == "sin_datos"].copy()
print("Productos sin datos nutricionales:", len(sin))
print(sin["grupo"].value_counts(), "\n")

partes = []
for g, d in sin.groupby("grupo"):
    partes.append(d.sample(n=max(1, round(25 * len(d) / len(sin))), random_state=42))
muestra = pd.concat(partes)

muestra["enlace"] = "https://tienda.mercadona.es/product/" + muestra["product_id"].astype(str)
muestra = muestra[["product_id", "nombre", "grupo", "enlace"]].sort_values("grupo")

os.makedirs("data/manual", exist_ok=True)
muestra.to_csv("data/manual/muestra_etiquetas.csv", index=False)
print(len(muestra), "productos en la muestra")
print(muestra[["product_id", "nombre", "grupo"]].to_string(index=False))

Productos sin datos nutricionales: 269
grupo
base               176
snack_dulce         74
plato_preparado     18
salsa                1
Name: count, dtype: int64 

26 productos en la muestra
 product_id                                                              nombre           grupo
      21896                                Aceitunas verdes sin hueso Hacendado            base
      11672              Queso semicurado de oveja Hacendado cortado en cuñitas            base
      22453                             Fideos orientales sabor pollo Hacendado            base
      50915                           Queso tierno bajo en sal mezcla Hacendado            base
      20033                                     Yogur sabor macedonia Hacendado            base
      21835                                Aceitunas verdes con hueso Hacendado            base
      35197                                             Embutido surtido oreado            base
      50942                             

In [9]:
silver_a = pd.read_csv("data/silver/productos_silver_ampliado.csv", dtype={"ean": str})

# Proteína real mirada en la etiqueta el 22/11/2026 (g por 100 g); 17925 no tiene etiqueta
reales = {
    21896: 1, 11672: 27, 22453: 2, 50915: 21, 20033: 3.5, 21835: 1.3, 35197: 13,
    50942: 23.2, 21549: 18.7, 50478: 22.6, 21450: 7.2, 58278: 31.9, 82117: 1.1,
    18341: 18, 6044: 15, 15109: 25.6, 18691: 8.26, 85827: 3.9, 3852: 1.9,
    65721: 6, 84289: 7.2, 85787: 4.1, 14151: 6.3, 17951: 3.9, 15430: 0.5,
}

sm = silver_a[silver_a["product_id"].isin(reales)].copy()
sm["real"] = sm["product_id"].map(reales)
sm["categoria_id"] = sm["categoria_id"].astype(str)
sm["formato"] = sm["formato"].fillna("desconocido")
sm["es_hacendado"] = sm["nombre"].str.contains("hacendado", case=False).astype(int)
sm["log_precio_ref"] = np.log1p(sm["precio_kg"])
sm["log_precio_ud"] = np.log1p(sm["precio_unidad"])
sm["log_tamano"] = np.log1p(sm["tamano_envase"])

# Predicciones con los modelos entrenados con todos los datos del Gold
sm["pred_media_cat"] = predecir_baseline(datos, sm)
for nombre_m, f in modelos.items():
    sm["pred_" + nombre_m.replace(" ", "_").lower()] = f().fit(datos, datos[OBJETIVO]).predict(sm)

# Los instantáneos se miran aparte (no sé si la etiqueta es del producto seco o preparado)
aparte = sm[sm["estado"] == "instantaneo"]
sm = sm[sm["estado"] != "instantaneo"].copy()

cols_pred = ["pred_media_cat", "pred_ridge", "pred_random_forest"]
print("Productos en el resultado principal:", len(sm), "| instantáneos aparte:", len(aparte))
print(sm[["product_id", "nombre", "grupo", "real"] + cols_pred].round(1).to_string(index=False))

print("\nError medio (g) y % de productos con error menor de 2 g:")
for c in cols_pred:
    err = (sm[c] - sm["real"]).abs()
    print(f"{c:20s} MAE = {err.mean():.2f} | error < 2 g: {(err < 2).mean() * 100:.0f} %")

print("\nInstantáneo aparte (real vs predicción):")
print(aparte[["nombre", "real"] + cols_pred].round(1).to_string(index=False))


# Solo los productos que recibirían estimación (categoría fiable)
fiab = pd.read_csv("resultados/fiabilidad_por_categoria.csv")
cats_fiables = set(fiab.loc[fiab["fiable"], "categoria_id"].astype(str))

sm["categoria_fiable"] = sm["categoria_id"].isin(cats_fiables)
sm["error_abs"] = (sm["pred_media_cat"] - sm["real"]).abs()

print("Productos de la muestra que recibirían estimación (categoría fiable):",
      int(sm["categoria_fiable"].sum()), "de", len(sm))

for nombre_g, d in [("CATEGORÍA FIABLE", sm[sm["categoria_fiable"]]),
                    ("CATEGORÍA NO FIABLE", sm[~sm["categoria_fiable"]])]:
    print(f"\n{nombre_g}: {len(d)} productos")
    if len(d):
        print(f"  Error medio: {d['error_abs'].mean():.2f} g")
        print(f"  Error menor de 2 g: {(d['error_abs'] < 2).mean() * 100:.0f} %")
        print(f"  Dentro de ±3,4 g: {(d['error_abs'] <= 3.4).sum()} de {len(d)} ({(d['error_abs'] <= 3.4).mean() * 100:.0f} %)")

print("\nProductos con estimación (categoría fiable):")
print(sm[sm["categoria_fiable"]][["nombre", "categoria_id", "real", "pred_media_cat", "error_abs"]]
      .round(1).sort_values("error_abs", ascending=False).to_string(index=False))

Productos en el resultado principal: 24 | instantáneos aparte: 1
 product_id                                                              nombre           grupo  real  pred_media_cat  pred_ridge  pred_random_forest
      21450 Cereales cubiertos de chocolate blanco rellenos con leche Hacendado            base   7.2             9.8        11.4                10.3
      14151                             Galletas con chocolate blanco Hacendado     snack_dulce   6.3             6.2         3.3                 6.1
      21896                                Aceitunas verdes sin hueso Hacendado            base   1.0             1.9        -2.3                 1.9
      21835                                Aceitunas verdes con hueso Hacendado            base   1.3             1.9        -2.9                 1.3
      18341                                      Nuez troceada pelada Hacendado            base  18.0            17.7        14.2                 8.6
      65721                      Su

### 8.2 Segunda muestra

Elijo 27 productos más al azar entre los sin datos que sí recibirían estimación (categoría fiable), sin repetir ninguno de los ya mirados.

In [10]:
silver_a = pd.read_csv("data/silver/productos_silver_ampliado.csv", dtype={"ean": str})
fiab = pd.read_csv("resultados/fiabilidad_por_categoria.csv")
cats_fiables = set(fiab.loc[fiab["fiable"], "categoria_id"].astype(str))

# Los 26 productos que ya miré a mano en la primera muestra
ya_mirados = {21896, 11672, 22453, 50915, 20033, 21835, 35197, 50942, 21549, 50478,
              21450, 58278, 82117, 18341, 6044, 15109, 18691, 85827, 3852, 17925,
              65721, 84289, 85787, 14151, 17951, 15430}

# Productos sin datos que sí recibirían estimación (categoría fiable), sin los ya mirados
pool = silver_a[
    (silver_a["estado_calidad"] == "sin_datos")
    & silver_a["categoria_id"].astype(str).isin(cats_fiables)
    & ~silver_a["product_id"].isin(ya_mirados)
    & (silver_a["estado"] != "instantaneo")
]
print("Productos disponibles para la segunda muestra:", len(pool))

nueva = pool.sample(n=min(27, len(pool)), random_state=7)
nueva["enlace"] = "https://tienda.mercadona.es/product/" + nueva["product_id"].astype(str)
nueva = nueva[["product_id", "nombre", "grupo", "enlace"]].sort_values("grupo")
nueva.to_csv("data/manual/muestra_etiquetas_2.csv", index=False)
print(nueva[["product_id", "nombre", "grupo"]].to_string(index=False))

Productos disponibles para la segunda muestra: 105
 product_id                                                    nombre       grupo
      21633                 Paté de hígado de cerdo ibérico Hacendado        base
      53160                       Sobrasada de Mallorca Can Pere Joan        base
      16807                                    Avena molida Hacendado        base
      11946          Aceitunas verdes rellenas de pepinillo Hacendado        base
      20846                                       Margarina Hacendado        base
      23217             Aceitunas verdes rellenas de anchoa Hacendado        base
      16272                         Paté de cerdo La Piara tapa negra        base
      82277                       Panecillo tostado classic Hacendado        base
      82287                             Pan tostado classic Hacendado        base
      58286      Jamón de bellota ibérico 50% La Hacienda del Ibérico        base
       6326                                    

### 8.3 Resultado conjunto: 40 productos de categorías fiables

Junto los 13 productos de la primera muestra que están en categorías fiables y los 27 de la segunda:

| | Test del experimento (187 productos) | Etiquetas reales (40 productos) |
|---|---|---|
| Error medio, media por categoría | 2,08 g | 2,41 g (intervalo 95 %: 1,67 a 3,25) |
| Error medio, Ridge | | 2,49 g |
| Error menor de 2 g | | 60 % |
| Dentro de ±3,4 g | 82,6 % | 80 % (32 de 40; intervalo 95 %: 65 % a 90 %) |

- El rango del 80 % se mantiene (32 de 40), con un intervalo amplio porque la muestra es pequeña.
- El error medio es algo mayor que en test, pero compatible con él. Lo razonable es decir «unos 2 a 2,5 g».
- Ridge vuelve a quedar igual que la media por categoría, así que sigo con la regla sencilla.
- Los 8 productos con error mayor de 3,4 g son sobre todo embutidos y platos especiales: longaniza de pavo (12 g), mini biscotes de trigo (9,2 g, la etiqueta marca 1,1 g), cocas de dacsa congeladas (8,0 g), paté de hígado (5,8 g) y torrezno (4,9 g).

In [11]:
# Proteína real de las etiquetas (g por 100 g), miradas el 22/11/2026
reales_1 = {
    21896: 1, 11672: 27, 20033: 3.5, 50915: 21, 21835: 1.3, 35197: 13,
    50942: 23.2, 21549: 18.7, 50478: 22.6, 21450: 7.2, 58278: 31.9, 82117: 1.1,
    18341: 18, 6044: 15, 15109: 25.6, 18691: 8.26, 85827: 3.9, 3852: 1.9,
    65721: 6, 84289: 7.2, 85787: 4.1, 14151: 6.3, 17951: 3.9, 15430: 0.5,
}
reales_2 = {
    21633: 8.5, 53160: 11, 16807: 12.2, 11946: 1.3, 20846: 0.5, 23217: 1.5,
    16272: 12, 82277: 11, 82287: 12, 58286: 29.5, 6326: 13, 58280: 29.5,
    21634: 15.1, 52622: 1, 18164: 17, 16280: 31, 82190: 8.6, 85702: 6,
    85713: 5.9, 85701: 6.2, 18035: 4.3, 67417: 4.3, 18752: 4.1, 82731: 6.3,
    18336: 5.6, 85709: 5.8, 85716: 7.1,
}
reales_todos = {**reales_1, **reales_2}

silver_a = pd.read_csv("data/silver/productos_silver_ampliado.csv", dtype={"ean": str})
fiab = pd.read_csv("resultados/fiabilidad_por_categoria.csv")
cats_fiables = set(fiab.loc[fiab["fiable"], "categoria_id"].astype(str))

m = silver_a[silver_a["product_id"].isin(reales_todos)].copy()
m["real"] = m["product_id"].map(reales_todos)
m["muestra"] = np.where(m["product_id"].isin(reales_2), "2ª", "1ª")
m["categoria_id"] = m["categoria_id"].astype(str)
m["formato"] = m["formato"].fillna("desconocido")
m["es_hacendado"] = m["nombre"].str.contains("hacendado", case=False).astype(int)
m["log_precio_ref"] = np.log1p(m["precio_kg"])
m["log_precio_ud"] = np.log1p(m["precio_unidad"])
m["log_tamano"] = np.log1p(m["tamano_envase"])

# Solo productos que recibirían estimación: categoría fiable y no instantáneos
m = m[m["categoria_id"].isin(cats_fiables) & (m["estado"] != "instantaneo")].copy()

m["pred_media_cat"] = predecir_baseline(datos, m)
m["pred_ridge"] = modelos["Ridge"]().fit(datos, datos[OBJETIVO]).predict(m)
m["error_media"] = (m["pred_media_cat"] - m["real"]).abs()
m["error_ridge"] = (m["pred_ridge"] - m["real"]).abs()

def wilson(k, n, z=1.96):
    p = k / n
    den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def resumen(d, etiqueta):
    n = len(d)
    k = int((d["error_media"] <= 3.4).sum())
    lo, hi = wilson(k, n)
    boot = [rng.choice(d["error_media"].values, size=n, replace=True).mean() for _ in range(5000)]
    print(f"{etiqueta}: {n} productos")
    print(f"  Error medio (media por categoría): {d['error_media'].mean():.2f} g "
          f"(intervalo 95 %: {np.percentile(boot, 2.5):.2f} a {np.percentile(boot, 97.5):.2f})")
    print(f"  Error medio de Ridge: {d['error_ridge'].mean():.2f} g")
    print(f"  Error < 2 g: {(d['error_media'] < 2).mean() * 100:.0f} %")
    print(f"  Dentro de ±3,4 g: {k} de {n} ({k / n * 100:.0f} %; intervalo 95 %: {lo * 100:.0f} % a {hi * 100:.0f} %)")

resumen(m, "TODOS (1ª + 2ª muestra)")
print()
resumen(m[m["muestra"] == "2ª"], "Solo 2ª muestra (elegida entre categorías fiables)")

print("\nProductos con error mayor de 3,4 g:")
print(m[m["error_media"] > 3.4][["nombre", "categoria_id", "real", "pred_media_cat", "error_media"]]
      .round(1).sort_values("error_media", ascending=False).to_string(index=False))

TODOS (1ª + 2ª muestra): 40 productos
  Error medio (media por categoría): 2.41 g (intervalo 95 %: 1.67 a 3.25)
  Error medio de Ridge: 2.49 g
  Error < 2 g: 60 %
  Dentro de ±3,4 g: 32 de 40 (80 %; intervalo 95 %: 65 % a 90 %)

Solo 2ª muestra (elegida entre categorías fiables): 27 productos
  Error medio (media por categoría): 2.13 g (intervalo 95 %: 1.36 a 3.14)
  Error medio de Ridge: 2.28 g
  Error < 2 g: 67 %
  Dentro de ±3,4 g: 23 de 27 (85 %; intervalo 95 %: 68 % a 94 %)

Productos con error mayor de 3,4 g:
                                                   nombre categoria_id  real  pred_media_cat  error_media
               Longaniza de pavo Imperial extra Hacendado           48  31.0            19.0         12.0
                         Mini biscotes de trigo Hacendado           62   1.1            10.3          9.2
                         Cocas de dacsa Susana congeladas          138   3.9            11.9          8.0
          Paté de hígado de cerdo sabor intenso Hacenda

## 9. Regla final: estimación por categoría y estado

El profesor sugirió empezar con medias por categoría y estado (base, snack dulce, plato preparado, salsa), con un mínimo de ejemplos compatibles. Aplico la misma regla de fiabilidad de la sección 6 (al menos 8 productos de entrenamiento y error de validación cruzada de 4 g o menos), pero ahora la clave del grupo es categoría + estado.

| | Solo categoría | Categoría + estado |
|---|---|---|
| Grupos fiables | 26 de 42 | 26 de 46 |
| Error medio en test | 2,08 g | 2,05 g |
| Rango del 80 % | ±3,4 g | −3,2 / +3,5 g |
| Cobertura en test | 82,6 % | 80,9 % |
| Productos sin datos con estimación | 120 de 269 | 115 de 269 |

- La precisión apenas cambia, pero el estado evita mezclar productos distintos dentro de una misma categoría. Por ejemplo, en legumbres la media de las secas es 8,8 g, la de las cocinadas 1,6 g y la de la categoría entera 5,5 g.
- Se pierden 5 estimaciones: el arroz seco tiene solo 7 productos (menos de 8) y los instantáneos solo 4, con una base incierta. Es lo que pidió el profesor.
- Esta es la regla que usará el frontal. Guarda `data/gold/estimaciones_proteina_por_estado.csv` y, en `resultados/`, los ficheros terminados en `_por_estado.csv`.

In [12]:
# --- Estimación por categoría Y estado, con las mismas reglas de fiabilidad de antes ---
def clave(df):
    # En arroz, legumbres y pasta el estado es seco, cocinado o fresco; en el resto vale "no_aplica"
    return df["categoria_id"].astype(str) + "|" + df["estado"].astype(str)

def predecir_por_clave(entrenamiento, df, con_respaldo=False):
    media = entrenamiento.groupby(clave(entrenamiento))[OBJETIVO].mean()
    p = clave(df).map(media).values.astype(float)
    if con_respaldo:   # solo para evaluar en validación: si falta el grupo, media de la categoría
        p = np.where(np.isnan(p), predecir_baseline(entrenamiento, df), p)
    return p

# 1) Errores fuera de pliegue en entrenamiento
oof_k = pd.Series(index=train.index, dtype=float)
for a_i, b_i in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
    oof_k.iloc[b_i] = predecir_por_clave(tr_f, va_f, con_respaldo=True)
res_k = train[OBJETIVO] - oof_k

# 2) Fiabilidad por grupo: al menos 8 productos y error medio <= 4 g
err_k = res_k.abs().groupby(clave(train)).agg(["mean", "count"])
fiables_k = err_k[(err_k["count"] >= 8) & (err_k["mean"] <= 4)].index
print("Grupos (categoría|estado) fiables:", len(fiables_k), "de", len(err_k))
print("\nGrupos de arroz, legumbres y pasta:")
print(err_k[err_k.index.str.split("|").str[0].isin(["118", "121", "120"])].round(2))

# 3) Rango de error del 80 % solo con grupos fiables
q10_k, q90_k = res_k[clave(train).isin(fiables_k)].quantile([0.1, 0.9])
print(f"\nRango de error (80 %): {q10_k:+.1f} g a {q90_k:+.1f} g   (antes: -3,4 g a +3,4 g)")

# 4) Comprobación en test
pred_k = predecir_por_clave(train, test)
mask_k = (clave(test).isin(fiables_k) & ~np.isnan(pred_k)).values
real = test[OBJETIVO].values
mae_k = np.abs(pred_k - real)[mask_k].mean()
cob_k = ((real >= pred_k + q10_k) & (real <= pred_k + q90_k))[mask_k].mean() * 100
print(f"\nTest, grupos fiables ({mask_k.sum()} productos): MAE {mae_k:.2f} g, cobertura del rango {cob_k:.1f} %")
print("   (antes, solo por categoría: 115 productos, MAE 2,08 g, cobertura 82,6 %)")

# 5) Aplicación a los productos sin datos nutricionales
sin_k = silver_amp[silver_amp["estado_calidad"] == "sin_datos"].copy()
sin_k["categoria_id"] = sin_k["categoria_id"].astype(str)
sin_k["estado"] = sin_k["estado"].astype(str)
media_k = datos.groupby(clave(datos))[OBJETIVO].mean()
sin_k["proteina_estimada"] = clave(sin_k).map(media_k).round(1)
sin_k["fiable"] = clave(sin_k).isin(fiables_k) & sin_k["proteina_estimada"].notna()
sin_k["rango_bajo"] = (sin_k["proteina_estimada"] + q10_k).clip(lower=0).round(1)
sin_k["rango_alto"] = (sin_k["proteina_estimada"] + q90_k).round(1)
sin_k.loc[~sin_k["fiable"], ["proteina_estimada", "rango_bajo", "rango_alto"]] = np.nan

print("\nProductos sin datos:", len(sin_k), "| con estimación fiable:", int(sin_k["fiable"].sum()),
      "| sin estimación:", int((~sin_k["fiable"]).sum()), "   (antes: 120 con estimación)")
cob_k_df = sin_k.groupby("grupo")["fiable"].agg(con_estimacion="sum", total="count")
print(cob_k_df)

print("\nProductos sin datos de arroz, legumbres y pasta, por estado:")
rlp = sin_k[sin_k["categoria_id"].isin(["118", "121", "120"])]
print(rlp.groupby(["categoria_id", "estado"])["fiable"].agg(con_estimacion="sum", total="count"))

sin_k[["product_id", "ean", "nombre", "categoria_id", "estado", "grupo",
       "proteina_estimada", "rango_bajo", "rango_alto", "fiable"]].to_csv(
    "data/gold/estimaciones_proteina_por_estado.csv", index=False)
tabla_k = err_k.round(2).rename(columns={"mean": "error_medio_g", "count": "productos"})
tabla_k["fiable"] = tabla_k.index.isin(fiables_k)
tabla_k.sort_values("error_medio_g").to_csv("resultados/fiabilidad_por_categoria_y_estado.csv")
cob_k_df.to_csv("resultados/cobertura_estimaciones_por_estado.csv")

Grupos (categoría|estado) fiables: 26 de 46

Grupos de arroz, legumbres y pasta:
                 mean  count
118|cocinado     4.45      2
118|seco         1.29      7
120|fresco       1.80      9
120|instantaneo  7.76      4
120|seco         1.99     29
121|cocinado     1.60      7
121|seco         8.83      6

Rango de error (80 %): -3.2 g a +3.5 g   (antes: -3,4 g a +3,4 g)

Test, grupos fiables (115 productos): MAE 2.05 g, cobertura del rango 80.9 %
   (antes, solo por categoría: 115 productos, MAE 2,08 g, cobertura 82,6 %)

Productos sin datos: 269 | con estimación fiable: 115 | sin estimación: 154    (antes: 120 con estimación)
                 con_estimacion  total
grupo                                 
base                         75    176
plato_preparado               2     18
salsa                         0      1
snack_dulce                  38     74

Productos sin datos de arroz, legumbres y pasta, por estado:
                          con_estimacion  total
categoria_id e

## 10. Conclusión

- **Regla elegida:** media de proteína por categoría y estado, calculada solo con los productos de entrenamiento. Ridge y Random Forest no mejoran de forma clara a esta regla sencilla, así que no los uso para estimar.
- **Error esperado:** unos 2 a 2,5 g de proteína por cada 100 g. En test es de 2,05 g y en las etiquetas reales que miré a mano es de 2,41 g.
- **Rango del 80 %:** entre −3,2 y +3,5 g. En las etiquetas reales cae dentro el 80 % de los productos (32 de 40), con un intervalo amplio porque la muestra es pequeña.
- **Cobertura:** estimo 115 de los 269 productos sin datos. En el resto (grupos con menos de 8 productos o con error de validación cruzada mayor de 4 g) no doy estimación.
- **Fuga corregida:** la mediana de `precio_kg` se calcula ahora dentro del pipeline, solo con el entrenamiento de cada fold. Los resultados no cambian a dos decimales.
- **Límites:** el test nuevo (101 productos) usa un mapeo de categorías que decidí antes de ver los resultados, y la comprobación con etiquetas reales es una muestra de 40 productos. Las estimaciones son orientativas y el frontal debe mostrarlas junto a su rango de error.
- **Fuera del ranking por euro:** los productos en litros y los instantáneos de base incierta (50 productos del Gold) no entran en el ranking de proteína por euro.